1.Importing os and Pandas for file and Data manipulation

In [67]:
import os
import pandas as pd

2. Creating new directories for file store

In [68]:
os.makedirs("../data/cleaned", exist_ok=True)

3. Loading Raw dataset for Data cleaning

In [69]:
orders_cleand = pd.read_csv("../data/raw/orders_raw.csv")

4. Converting text column into string dtypes and removing extra whitespace leading and trailing from the text

In [70]:
text_column = [
    "order_id",
    "customer_id",
    "product_id",
    "payment_method",
    "order_status",
]

for col in text_column:
    orders_cleand[col] = orders_cleand[col].astype(str).str.strip()

5. Replaced missing 'customer_id' to unknown

In [71]:
orders_cleand["customer_id"] = orders_cleand["customer_id"].replace("NaN","Unknown")

6.Remove negative order quantity to potive by absolutr method

In [72]:
orders_cleand["quantity"] = orders_cleand["quantity"].abs().astype(str)

6. Impute 300 missing customer ratings with the dataset median rating

In [73]:
median_rating = orders_cleand["customer_rating"].median()
orders_cleand["customer_rating"] = orders_cleand["customer_rating"].fillna(median_rating)

7. Convert order_date dtype to datetime dtype

In [74]:
orders_cleand["order_date"] =pd.to_datetime(orders_cleand["order_date"], errors="coerce")

8. Convert int, float datetype to numeric dtype based on the input dataypes in the column for validation

In [75]:
numeric_cols = [
    "quantity",
    "unit_price",
    "discount_pct",
    "shipping_cost",
    "estimated_cost"
]

for col in numeric_cols:
    orders_cleand[col] = pd.to_numeric(
        orders_cleand[col],
        errors="coerce"
    )

9. Calculate revenue and profit to make sure the error handling to ensure exact precision with positive quantities

In [76]:
orders_cleand["revenue"] = (
    orders_cleand["quantity"]
    * orders_cleand["unit_price"]
    * (1 - orders_cleand["discount_pct"])
).round(2)

orders_cleand["profit"] = (
    orders_cleand["revenue"]
    - orders_cleand["shipping_cost"]
    - orders_cleand["estimated_cost"]
).round(2)

10. Standardize payemnt method text into same standard 

In [77]:
orders_cleand["payment_method"] = orders_cleand["payment_method"].str.title()
orders_cleand["payment_method"] = orders_cleand["payment_method"].replace({
    "Upi": "UPI",
    "Cod": "COD",
    "credit card" : "credit Card",
    "debit card" : "Debit Card",
    "net banking" : "Net Banking",
    "wallet" : "Wallet",
})

11. Verifying the standardization

In [78]:
print(orders_cleand["payment_method"].value_counts())

payment_method
Net Banking    8513
UPI            8385
Credit Card    8372
Wallet         8287
COD            8270
Debit Card     8173
Name: count, dtype: int64


12. Verifying the shape, missing values, negative quantities, data types before export to csv

In [79]:
print("Dataset Shape:", orders_cleand.shape)
print("Remaining Missing Values:\n", orders_cleand.isnull().sum())
print("Remaining Negative Quantities:", (orders_cleand["quantity"] < 0).sum())
print("Data Types:\n", orders_cleand.dtypes)

Dataset Shape: (50000, 14)
Remaining Missing Values:
 order_id           0
customer_id        0
product_id         0
order_date         0
quantity           0
unit_price         0
discount_pct       0
revenue            0
shipping_cost      0
payment_method     0
order_status       0
customer_rating    0
estimated_cost     0
profit             0
dtype: int64
Remaining Negative Quantities: 0
Data Types:
 order_id                   object
customer_id                object
product_id                 object
order_date         datetime64[ns]
quantity                    int64
unit_price                float64
discount_pct              float64
revenue                   float64
shipping_cost             float64
payment_method             object
order_status               object
customer_rating           float64
estimated_cost            float64
profit                    float64
dtype: object


13. Export the result after cleaning to csv file in '../data/cleaned' folder

In [80]:
output_path = os.path.join("../data/cleaned", "orders_cleaned.csv")
orders_cleand.to_csv(output_path, index= False)